# Lab Day 2 — Backbone, Training Recipe & Inference on DeepWeeds
**Student:** Doan Quang Minh (2A202602711)
**GPU:** NVIDIA GeForce RTX 4060 Laptop

**Quy tắc quan trọng:** Chọn mọi thứ trên **val**; **test chỉ chạy một lần mỗi seed ở Bước 4**.

## 0. Cài đặt môi trường

In [ ]:
import sys
import os
import platform
import numpy as np
import pandas as pd
import torch
import timm
import matplotlib.pyplot as plt

# Thêm đường dẫn để import code
CODE_DIR = "submissions/2A202602711_DoanQuangMinh/code"
REPO_DIR = "D:/K4-Track4-Day2-Deeplearning-Advance"
sys.path.insert(0, CODE_DIR)
sys.path.insert(0, REPO_DIR)

print(f"Python {platform.python_version()}")
print(f"PyTorch {torch.__version__}")
print(f"timm {timm.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

### Đường dẫn dữ liệu

In [ ]:
# Đường dẫn dữ liệu - CẬP NHẬT THEO MÁY CỦA BẠN
IMAGES_DIR = "D:/K4-Track4-Day2-Deeplearning-Advance/data/images"
LABELS_DIR = "D:/K4-Track4-Day2-Deeplearning-Advance/data/labels"

# Verify paths
print(f"Images: {IMAGES_DIR}")
print(f"Labels: {LABELS_DIR}")
n_images = len([f for f in os.listdir(IMAGES_DIR) if f.endswith('.jpg')])
print(f"Number of images: {n_images}")
print(f"Expected: 17509")

---
## Bước 0 — EDA và Kiểm tra Pipeline

### 0.1 Đọc và kiểm tra Split

In [ ]:
from dataset import load_split, check_split

# Load split (fold 0)
train_df, val_df, test_df = load_split(LABELS_DIR, fold=0)
print(f"Loaded: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")

# Verify split
stats = check_split(train_df, val_df, test_df, IMAGES_DIR)

In [ ]:
# Tính ratio lớn nhất / nhỏ nhất
train_counts = train_df['Label'].value_counts().sort_index()
max_count = train_counts.max()
min_count = train_counts[train_counts > 0].min()
print(f"\nClass imbalance ratio (max/min): {max_count/min_count:.2f}x")
print(f"Negative class: {train_counts[8]} images ({train_counts[8]/len(train_df)*100:.1f}%)")

### 0.2 Biểu đồ phân bố lớp

In [ ]:
CLASS_NAMES = [
    "Chinee Apple", "Lantana", "Parkinsonia", "Parthenium",
    "Prickly Acacia", "Rubber Vine", "Siam Weed", "Snake Weed", "Negatives"
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Train
train_counts = train_df['Label'].value_counts().sort_index()
axes[0].bar(range(9), [train_counts.get(i, 0) for i in range(9)])
axes[0].set_xticks(range(9))
axes[0].set_xticklabels([n[:8] for n in CLASS_NAMES], rotation=45, ha='right')
axes[0].set_title(f'Train (n={len(train_df)})')
axes[0].set_ylabel('Count')

# Val
val_counts = val_df['Label'].value_counts().sort_index()
axes[1].bar(range(9), [val_counts.get(i, 0) for i in range(9)])
axes[1].set_xticks(range(9))
axes[1].set_xticklabels([n[:8] for n in CLASS_NAMES], rotation=45, ha='right')
axes[1].set_title(f'Val (n={len(val_df)})')

# Test
test_counts = test_df['Label'].value_counts().sort_index()
axes[2].bar(range(9), [test_counts.get(i, 0) for i in range(9)])
axes[2].set_xticks(range(9))
axes[2].set_xticklabels([n[:8] for n in CLASS_NAMES], rotation=45, ha='right')
axes[2].set_title(f'Test (n={len(test_df)})')

plt.tight_layout()
plt.savefig('curves/eda_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to curves/eda_class_distribution.png")

### 0.3 Xem mẫu ảnh từng lớp

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

fig, axes = plt.subplots(9, 3, figsize=(8, 24))
for label in range(9):
    samples = train_df[train_df['Label'] == label].head(3)
    for j, (_, row) in enumerate(samples.iterrows()):
        img_path = os.path.join(IMAGES_DIR, row['Filename'])
        img = Image.open(img_path)
        axes[label, j].imshow(img)
        axes[label, j].axis('off')
        if j == 0:
            axes[label, j].set_ylabel(CLASS_NAMES[label][:10], fontsize=8)
plt.suptitle('Sample images per class (3 each)')
plt.tight_layout()
plt.savefig('curves/eda_sample_images.png', dpi=100, bbox_inches='tight')
plt.show()

---
### 0.4 Kiểm tra Pipeline

In [ ]:
print("="*60)
print("PIPELINE CHECKS")
print("="*60)

# Check 1: Seed cố định
from train import set_seed
set_seed(42)
a = np.random.rand()
set_seed(42)
b = np.random.rand()
seed_check = np.allclose(a, b)
print(f"1. Seed fixed: {'PASS' if seed_check else 'FAIL'}")

In [ ]:
# Check 2: Initial loss
from model import build_model
import torch.nn.functional as F

model_scratch = build_model('resnet50', pretrained=False, num_classes=9, init='scratch')
model_scratch = model_scratch.cuda()
model_scratch.eval()
with torch.no_grad():
    dummy = torch.randn(8, 3, 224, 224).cuda()
    out = model_scratch(dummy)
    init_loss = F.cross_entropy(out, torch.zeros(8, dtype=torch.long).cuda())
print(f"2. Initial CE loss: {init_loss.item():.4f} (expected ~2.197)")
print(f"   {'PASS' if abs(init_loss.item() - 2.197) < 0.5 else 'CHECK'}")

In [ ]:
# Check 3: Overfit tiny batch
from dataset import build_transforms, DeepWeedsDataset
from losses import build_criterion
from torch.utils.data import DataLoader

tiny_df = train_df.head(4)
transform = build_transforms(train=True, img_size=224)
dataset = DeepWeedsDataset(tiny_df, IMAGES_DIR, transform)
loader = DataLoader(dataset, batch_size=4)

model = build_model('resnet50', pretrained=True, num_classes=9, init='finetune')
model = model.cuda()
model.train()
criterion = build_criterion('ce')
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for i in range(50):
    for images, labels, _ in loader:
        images, labels = images.cuda(), labels.cuda()
        optimizer.zero_grad()
        out = model(images)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()

with torch.no_grad():
    for images, labels, _ in loader:
        images, labels = images.cuda(), labels.cuda()
        out = model(images)
        final_loss = criterion(out, labels)
        preds = out.argmax(dim=1)
        acc = (preds == labels).float().mean()

print(f"3. Overfit tiny batch: loss={final_loss.item():.4f}, acc={acc.item():.2%}")
print(f"   {'PASS' if final_loss.item() < 0.5 else 'CHECK'}")

In [ ]:
# Check 4: Focal loss gamma=0 equals CE
from losses import FocalLoss

logits = torch.randn(32, 9)
target = torch.randint(0, 9, (32,))
ce_loss = F.cross_entropy(logits, target)
focal_loss = FocalLoss(gamma=0.0)
f_loss = focal_loss(logits.cuda().float(), target.cuda())
diff = abs(ce_loss.item() - f_loss.cpu().item())
print(f"4. Focal(gamma=0) = CE: diff={diff:.2e}")
print(f"   {'PASS' if diff < 1e-4 else 'FAIL'}")

In [ ]:
# Check 5: Mixup/CutMix work
from losses import mix_batch

x = torch.randn(4, 3, 224, 224)
y = torch.randint(0, 9, (4,))

# Mixup
x_mix, (y_a, y_b, lam) = mix_batch(x, y, alpha=1.0, mode='mixup')
print(f"5. Mixup: x_mix shape={x_mix.shape}, lam={lam:.3f} - PASS")

# CutMix
x_mix, (y_a, y_b, lam) = mix_batch(x, y, alpha=1.0, mode='cutmix')
print(f"   CutMix: x_mix shape={x_mix.shape}, lam={lam:.3f} - PASS")

In [ ]:
# Check 6: Param groups
from model import param_groups

model = build_model('resnet50', pretrained=True, num_classes=9, init='finetune')
groups = param_groups(model, lr_backbone=1e-4, lr_head=1e-3, weight_decay=0.05)
print(f"6. Param groups: {len(groups)} groups")
for i, g in enumerate(groups):
    print(f"   Group {i+1}: {len(g['params'])} params, lr={g['lr']}, wd={g['weight_decay']}")
print(f"   {'PASS' if len(groups) == 3 else 'CHECK'}")

In [ ]:
# Check 7: Frozen BN stays in eval mode
from model import freeze_backbone, set_bn_eval

model = build_model('resnet50', pretrained=True, num_classes=9, init='frozen')
freeze_backbone(model)
model.train()  # Should not affect frozen BN
set_bn_eval(model)  # Explicitly set BN to eval

bn_ok = True
for m in model.modules():
    if isinstance(m, torch.nn.BatchNorm2d):
        if m.training:
            bn_ok = False
            print(f"   FAIL: {m}")
print(f"7. Frozen BN stays in eval: {'PASS' if bn_ok else 'FAIL'}")

---
## Bước 1 — So sánh Backbone

**NOTE:** Chạy từ terminal với:
```
python code/train.py --set exp_id=B01 backbone=resnet50 seed=0 epochs=12
```

Các backbone cần test:
- B01: ResNet50
- B02: EfficientNet-B0
- B03: ConvNeXt-Tiny
- B04: DeiT-Small
- B05: Swin-Tiny
- B06: MobileNetV3

## Bước 2 — Công thức Huấn luyện

**NOTE:** Chạy từ terminal với các biến thể augmentation, loss, etc.

## Bước 3 — Inference Techniques

**NOTE:** Inference sẽ được implement trong inference.py

## Bước 4 — Final Run

Chạy 3 seeds với config tốt nhất:
```
python code/train.py --set exp_id=F01 backbone=xxx seed=0 epochs=12 save_test_predictions=True
python code/train.py --set exp_id=F01 backbone=xxx seed=1 epochs=12 save_test_predictions=True
python code/train.py --set exp_id=F01 backbone=xxx seed=2 epochs=12 save_test_predictions=True
```

Sau đó chạy eval.py:
```bash
python eval.py score --pred "predictions/F01_seed*_test.csv" --test-csv data/labels/test_subset0.csv --labels data/labels/labels.csv --tag F01
python eval.py grade --final "predictions/F01_seed*_test.csv" --baseline "predictions/T00_seed*_test.csv" --test-csv data/labels/test_subset0.csv --labels data/labels/labels.csv
```

In [ ]:
# Verify eval.py works on smoke test predictions
import subprocess
result = subprocess.run([
    'python', 'eval.py', 'score',
    '--pred', 'predictions/B01_seed0_val.csv',
    '--test-csv', f'{LABELS_DIR}/val_subset0.csv',
    '--labels', f'{LABELS_DIR}/labels.csv',
    '--tag', 'B01'
], capture_output=True, text=True)
print(result.stdout)